# 🐢 Tortuga — Treino YOLOv8s no Google Colab
**Challenge SPI 2026 — FIAP × SPI**

Este notebook treina o modelo de detecção de EPIs do Tortuga usando:
- **YOLOv8s** (modelo médio — melhor generalização)
- **~52k imagens** de 3 datasets do Roboflow Universe
- **100 épocas** com early stopping

**GPU recomendada: A100** (2-3h de treino, ~20-25 compute units)  
**RAM recomendada: Alta RAM (25 GB)** — evita OOM no carregamento do dataset

> Altere em: Runtime → Change runtime type → A100 + High RAM

In [ ]:
# ── Célula 1: Verifica GPU e RAM disponíveis ──────────────────
import subprocess

print('=== GPU ===')
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)
print(result.stdout.strip())

print('\n=== RAM ===')
with open('/proc/meminfo') as f:
    for line in f:
        if 'MemTotal' in line:
            kb = int(line.split()[1])
            print(f'RAM Total: {kb/1024/1024:.1f} GB')
            break

import torch
print(f'\nCUDA disponivel: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ── Célula 2: Instala dependências ────────────────────────────
!pip install -q ultralytics roboflow
print('Instalação concluída!')

In [ ]:
# ── Célula 3: Monta Google Drive (para salvar o modelo) ───────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/Tortuga/models'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Modelos serão salvos em: {DRIVE_OUTPUT}')

In [ ]:
# ── Célula 4: Configuração — coloque sua API Key aqui ─────────
ROBOFLOW_API_KEY = 'LEPf043TOdrTdQHF4bf6'  # sua chave Roboflow

import os
os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY
os.environ['YOLO_OFFLINE'] = '0'  # precisa de internet para baixar datasets

DATASET_DIR = '/content/datasets'
os.makedirs(DATASET_DIR, exist_ok=True)
print('Config OK')

In [ ]:
# ── Célula 5: Download dos 3 datasets do Roboflow ─────────────
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

DATASETS = [
    {
        'name': 'ppe-combined-44k',
        'workspace': 'roboflow-universe-projects',
        'project': 'personal-protective-equipment-combined-model',
        'version': 8,
    },
    {
        'name': 'safety-ppe-shoes',
        'workspace': 'safety-jmser',
        'project': 'safety_ppe',
        'version': 23,
    },
    {
        'name': 'hearing-protection',
        'workspace': 'siabar',
        'project': 'ppe-dataset-for-workplace-safety',
        'version': 1,
    },
]

for ds in DATASETS:
    dest = f"{DATASET_DIR}/{ds['name']}"
    if os.path.exists(dest):
        print(f"  Ja existe: {ds['name']} — pulando")
        continue
    print(f"\n  Baixando: {ds['name']}...")
    try:
        project = rf.workspace(ds['workspace']).project(ds['project'])
        version = project.version(ds['version'])
        version.download('yolov8', location=dest)
        print(f'  OK: {dest}')
    except Exception as e:
        print(f'  ERRO em {ds["name"]}: {e}')

print('\nDownload concluído!')

In [ ]:
# ── Célula 6: Merge de datasets com remapeamento de classes ───
import shutil, yaml
from pathlib import Path

TARGET_CLASSES = [
    'Hardhat',         'NO-Hardhat',
    'Safety Vest',     'NO-Safety Vest',
    'Safety Glasses',  'NO-Safety Glasses',
    'Mask',            'NO-Mask',
    'Ear Protection',  'NO-Ear Protection',
    'Safety Boots',    'NO-Safety Boots',
]

CLASS_REMAP = {
    # Protetor auricular
    'ear protection': 'Ear Protection', 'ear_protection': 'Ear Protection',
    'ear-protection': 'Ear Protection', 'hearing protection': 'Ear Protection',
    'earmuff': 'Ear Protection', 'earmuffs': 'Ear Protection',
    'no ear protection': 'NO-Ear Protection', 'no_ear_protection': 'NO-Ear Protection',
    'no-ear-protection': 'NO-Ear Protection',
    # Bota
    'safety boots': 'Safety Boots', 'safety_boots': 'Safety Boots',
    'safety shoes': 'Safety Boots', 'safety_shoes': 'Safety Boots',
    'boots': 'Safety Boots', 'boot': 'Safety Boots',
    'shoe': 'Safety Boots', 'shoes': 'Safety Boots',
    'no safety boots': 'NO-Safety Boots', 'no_safety_boots': 'NO-Safety Boots',
    'no_shoe': 'NO-Safety Boots', 'no shoe': 'NO-Safety Boots',
    'regular shoes': 'NO-Safety Boots', 'regular_shoes': 'NO-Safety Boots',
    # Capacete
    'hardhat': 'Hardhat', 'hard hat': 'Hardhat', 'helmet': 'Hardhat', 'hard-hat': 'Hardhat',
    'no hardhat': 'NO-Hardhat', 'no-hardhat': 'NO-Hardhat',
    'no_hardhat': 'NO-Hardhat', 'no helmet': 'NO-Hardhat', 'no_helmet': 'NO-Hardhat',
    # Colete
    'safety vest': 'Safety Vest', 'vest': 'Safety Vest', 'hi-vis': 'Safety Vest',
    'no safety vest': 'NO-Safety Vest', 'no-safety vest': 'NO-Safety Vest',
    'no_safety_vest': 'NO-Safety Vest', 'no vest': 'NO-Safety Vest',
    # Óculos / Goggles
    'safety glasses': 'Safety Glasses', 'glasses': 'Safety Glasses',
    'glass': 'Safety Glasses', 'goggles': 'Safety Glasses', 'goggle': 'Safety Glasses',
    'no glasses': 'NO-Safety Glasses', 'no_glasses': 'NO-Safety Glasses',
    'no safety glasses': 'NO-Safety Glasses',
    'no_goggles': 'NO-Safety Glasses', 'no-goggles': 'NO-Safety Glasses',
    'no goggles': 'NO-Safety Glasses',
    # Máscara
    'mask': 'Mask', 'face mask': 'Mask', 'respirator': 'Mask',
    'no mask': 'NO-Mask', 'no-mask': 'NO-Mask', 'no_mask': 'NO-Mask',
    # Ignorar
    'person': None, 'glove': None, 'gloves': None, 'no_glove': None,
    'worker': None, 'human': None, 'ladder': None, 'safety cone': None,
    'fall-detected': None, 'safety_harness': None, 'no_harness': None,
    'no_breathingapparatus': None,
}

merged_dir = Path(DATASET_DIR) / 'merged'
for split in ['train', 'valid', 'test']:
    (merged_dir / split / 'images').mkdir(parents=True, exist_ok=True)
    (merged_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

class_to_id = {c.lower(): i for i, c in enumerate(TARGET_CLASSES)}
img_count = {'train': 0, 'valid': 0, 'test': 0}

for ds_dir in Path(DATASET_DIR).iterdir():
    if ds_dir.name == 'merged' or not ds_dir.is_dir():
        continue
    yaml_file = ds_dir / 'data.yaml'
    if not yaml_file.exists():
        candidates = list(ds_dir.rglob('data.yaml'))
        if not candidates:
            print(f'  AVISO: {ds_dir.name} sem data.yaml — ignorando')
            continue
        yaml_file = candidates[0]
        ds_dir = yaml_file.parent
    with open(yaml_file) as f:
        ds_cfg = yaml.safe_load(f)
    ds_classes = [c.lower() for c in ds_cfg.get('names', [])]
    print(f'  Mesclando: {ds_dir.name} ({len(ds_classes)} classes)')
    for split in ['train', 'valid', 'test']:
        img_dir = ds_dir / split / 'images'
        lbl_dir = ds_dir / split / 'labels'
        if not img_dir.exists():
            continue
        for img_path in img_dir.glob('*.*'):
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue
            new_lines = []
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts: continue
                    old_id = int(parts[0])
                    if old_id >= len(ds_classes): continue
                    old_cls = ds_classes[old_id]
                    mapped = CLASS_REMAP.get(old_cls, '')
                    if mapped is None: continue
                    new_cls = mapped.lower() if mapped else old_cls
                    new_id = class_to_id.get(new_cls)
                    if new_id is None: continue
                    new_lines.append(f"{new_id} " + ' '.join(parts[1:]))
            if not new_lines: continue
            shutil.copy2(img_path, merged_dir / split / 'images' / img_path.name)
            with open(merged_dir / split / 'labels' / lbl_path.name, 'w') as f:
                f.write('\n'.join(new_lines))
            img_count[split] += 1

yaml_content = {
    'path': str(merged_dir), 'train': 'train/images', 'val': 'valid/images',
    'test': 'test/images', 'nc': len(TARGET_CLASSES), 'names': TARGET_CLASSES,
}
with open(merged_dir / 'data.yaml', 'w') as f:
    yaml.dump(yaml_content, f, allow_unicode=True)

print(f'\nDataset mesclado:')
for split, cnt in img_count.items():
    print(f'  {split}: {cnt} imagens')

In [ ]:
# ── Célula 7: Treino YOLOv8s ──────────────────────────────────
from ultralytics import YOLO
import torch

print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

model = YOLO('yolov8s.pt')

results = model.train(
    data=str(merged_dir / 'data.yaml'),
    epochs=100,
    batch=32,          # A100 tem VRAM suficiente para batch 32
    imgsz=640,
    device='cuda',
    project='/content/runs/train',
    name='tortuga_ppe_s',
    exist_ok=True,
    patience=25,
    save=True,
    val=True,
    plots=True,
    # Augmentação para ambientes variados
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.7,
    degrees=12.0,
    translate=0.15,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    erasing=0.4,
    auto_augment='randaugment',
)

print('\nTreino concluído!')
print(f'mAP50: {results.results_dict.get("metrics/mAP50(B)", 0):.3f}')

In [ ]:
# ── Célula 8: Salva modelo no Google Drive ────────────────────
import shutil
from pathlib import Path

best_pt = Path('/content/runs/train/tortuga_ppe_s/weights/best.pt')

if best_pt.exists():
    dest = Path(DRIVE_OUTPUT) / 'best_fase1_s.pt'
    shutil.copy2(best_pt, dest)
    print(f'Modelo salvo no Drive: {dest}')
    print(f'Tamanho: {dest.stat().st_size / 1024 / 1024:.1f} MB')
else:
    print('ERRO: best.pt nao encontrado!')
    print('Verifique: /content/runs/train/tortuga_ppe_s/weights/')

In [ ]:
# ── Célula 9: Validação final por classe ──────────────────────
from ultralytics import YOLO
from pathlib import Path

model = YOLO('/content/runs/train/tortuga_ppe_s/weights/best.pt')
metrics = model.val(data=str(merged_dir / 'data.yaml'), split='test')

print('\n=== Resultados por classe ===')
TARGET_CLASSES = [
    'Hardhat', 'NO-Hardhat', 'Safety Vest', 'NO-Safety Vest',
    'Safety Glasses', 'NO-Safety Glasses', 'Mask', 'NO-Mask',
    'Ear Protection', 'NO-Ear Protection', 'Safety Boots', 'NO-Safety Boots',
]
for i, cls in enumerate(TARGET_CLASSES):
    try:
        ap = metrics.box.ap50[i]
        print(f'  {cls:<22} mAP50: {ap:.3f}')
    except Exception:
        pass
print(f'\n  mAP50 geral: {metrics.box.map50:.3f}')